In [ ]:
import pandas as pd

llava_annotations = pd.read_csv(snakemake.input.cellwhisperer_labels)
system_message = snakemake.params.request

In [ ]:
# Try OpenAI first if key is provided
openai_client = None
openai_api_key = snakemake.params.openai_api_key
if openai_api_key:
    from openai import OpenAI
    client = OpenAI(api_key=openai_api_key)
    try:
        client.models.list()
        openai_client = client
        print("Using OpenAI API for curation")
    except Exception as e:
        print(f"OpenAI API key invalid ({type(e).__name__}), falling back to local model")

In [ ]:
# Load local model if OpenAI is unavailable
local_pipeline = None
if openai_client is None:
    from transformers import pipeline
    import torch

    model_name = snakemake.params.local_model
    print(f"Loading local model: {model_name}")
    local_pipeline = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    print("Local model loaded")

In [ ]:
results = {}
for idx, row in llava_annotations.iterrows():
    if not isinstance(row["cluster_annotations"], str):
        response = "No label"
    elif openai_client is not None:
        chat_completion = openai_client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": row["cluster_annotations"]},
            ],
            model="gpt-4-turbo-preview",
            max_tokens=20,
            temperature=0,
        )
        response = chat_completion.choices[0].message.content
    else:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": row["cluster_annotations"]},
        ]
        output = local_pipeline(
            messages,
            max_new_tokens=20,
            temperature=0.1,
            do_sample=True,
        )
        response = output[0]["generated_text"][-1]["content"].strip()

    print(f"Cluster {row['cluster_values']}: {response}")
    results[idx] = response

In [ ]:
llava_annotations["curated_labels"] = results
llava_annotations.to_csv(snakemake.output.curated_labels, index=False)